# NB5 · Instructor: prepare the judge cache

**MICAI 2026 · Mechanistic Interpretability, Hands-on** · Block 5 *Audit: can we trust the judge?* (Víctor Macías-Terrón)
**Who runs this:** a presenter, **once, before the session** (not the participants).
**Runtime:** Colab **T4 GPU** for `gemma3-4b` (≈ 10–20 min including the ≈ 8 GB download; see the timing printouts). `qwen0.5b` runs on CPU (measured: 32 min on 2 shared cores, of which 16 min is the verbalized-confidence generation).

**What it does.** Runs the LLM judge over a fixed selection of LegalBench `contract_nli_*` examples and saves a compact cache that NB5 loads, so participants need no GPU:

| field | meaning |
|---|---|
| `uid, task, assertion, excerpt, split, gold` | the example; `split ∈ {train, cal, test, shift}` (`shift` = held-out assertions) |
| `p_yes`, `yes_no_mass` | next-token P(Yes)/(P(Yes)+P(No)) at the answer position; mass on {Yes, No} (sanity check) |
| `verbal_conf`, `verbal_raw` | the judge's own 0–100 confidence in its verdict (second, short generation) |
| `hidden`, `layers` | residual stream at the last prompt token, selected layers, float16 `[N, L, d_model]` |
| `g_*` | precomputed **grounding check** for confident *Yes* verdicts (evidence replaced / one segment removed) |

**Steps for the presenters**
1. Runtime → Change runtime type → **T4 GPU**. Add your HF token as Colab secret `HF_TOKEN` (accept the Gemma 3 licence on its model page first).
2. Fill in `CACHE_REPO` below (a **dataset** repo you can write to, e.g. `your-user/micai2026-nb5-judge-cache`). Leave `UPLOAD = True`.
3. Run all. Check the sanity printout (judge accuracy should be well above 50 %, Yes/No mass close to 1, no NaNs).
4. Put the same `CACHE_REPO` string in the student notebook generator (`notebooks/src/build_nb5.py`, `CACHE_CONSTANTS`) and rebuild; make the repo **public** so participants do not need a token to read it.

Alternative without HF: download the `.npz` (Files panel) and host it on Google Drive with a public direct-download link; put that link in `FALLBACK_URL`.

In [ ]:
%pip install -q transformers==5.20.2 huggingface_hub==2.2.0 datasets==5.1.0

## 0 · Configuration

In [ ]:
# ─── TODO (presenters): publish the cache, then fill these in ─────────────────────────────
CACHE_REPO   = "TODO-your-hf-user/micai2026-nb5-judge-cache"   # HF *dataset* repo with cache_<model>.npz
FALLBACK_URL = "https://raw.githubusercontent.com/salvahin/mechinterp-tutorial/main/data/nb5/cache_qwen0.5b.npz"
# ──────────────────────────────────────────────────────────────────────────────────────────

MODEL_CHOICE = "gemma3-4b"        # session judge (T4).  "qwen0.5b" = ungated fallback, runs on CPU
UPLOAD = not CACHE_REPO.startswith("TODO")
N_PER_ASSERTION = 40              # balanced Yes/No where the data allows -> 540 examples (same selection for both models)
SEED = 2026
MAX_EXCERPT_CHARS = 1000          # long excerpts are truncated at a word boundary (keeps prompts < ~350 tokens)
DO_VERBAL = True                  # verbalized confidence (second short generation)
N_GROUNDING = 40                  # most confident Yes verdicts (test + shift) for the grounding check
OUT_DIR = "nb5_cache"

In [ ]:
import os, sys, time, json, hashlib, importlib.util
import numpy as np, pandas as pd, torch

IN_COLAB = "google.colab" in sys.modules or importlib.util.find_spec("google.colab") is not None
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))
BATCH_SIZE = 8 if device == "cuda" else 4
print("device:", device, "| model:", MODEL_CHOICE, "| batch:", BATCH_SIZE)
if MODEL_CHOICE.startswith("gemma") and device == "cpu":
    print("⚠ Gemma 3 4B on CPU is very slow (and needs ~10 GB RAM). Use a T4 runtime.")
os.makedirs(OUT_DIR, exist_ok=True)

## 1 · Hugging Face login (Gemma 3 is gated; uploading needs write access)

In [ ]:
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"No usable Colab secret HF_TOKEN ({type(e).__name__}).")
if not HF_TOKEN:
    from huggingface_hub import get_token
    HF_TOKEN = os.environ.get("HF_TOKEN") or get_token()
if not HF_TOKEN and (MODEL_CHOICE.startswith("gemma") or UPLOAD):
    from huggingface_hub import notebook_login
    notebook_login()
    from huggingface_hub import get_token
    HF_TOKEN = get_token()
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
print("token:", "found" if HF_TOKEN else "none (fine for qwen0.5b without upload)")

## 2 · Task table and example selection

In [ ]:
# LegalBench (Guha et al., NeurIPS 2023) contract_nli_* tasks on HF `nguha/legalbench`, all CC BY 4.0,
# derived from ContractNLI (Koreeda & Manning, 2021). One assertion (= our "rule") per task.
# Assertion text copied from each task's README (LegalBench GitHub); "ome" -> "some" typo fixed.
ASSERTIONS = {
    "confidentiality_of_agreement": "the Receiving Party shall not disclose the fact that Agreement was agreed or negotiated",
    "explicit_identification": "all Confidential Information shall be expressly identified by the Disclosing Party",
    "inclusion_of_verbally_conveyed_information": "Confidential Information may include verbally conveyed information",
    "limited_use": "the Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in Agreement",
    "no_licensing": "the Agreement shall not grant Receiving Party any right to Confidential Information",
    "notice_on_compelled_disclosure": "the Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regulation or judicial process to disclose any Confidential Information",
    "permissible_acquirement_of_similar_information": "the Receiving Party may acquire information similar to Confidential Information from a third party",
    "permissible_copy": "the Receiving Party may create a copy of some Confidential Information in some circumstances",
    "permissible_development_of_similar_information": "the Receiving Party may independently develop information similar to Confidential Information",
    "permissible_post-agreement_possession": "the Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information",
    "return_of_confidential_information": "the Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement",
    "sharing_with_employees": "the Receiving Party may share some Confidential Information with some of Receiving Party's employees",
    "sharing_with_third-parties": "the Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors)",
    "survival_of_obligations": "some obligations of Agreement may survive termination of Agreement",
}
# Held out entirely (never seen by the probe): stands in for "a new 29110 process" -> distribution shift
HELDOUT = ["notice_on_compelled_disclosure", "return_of_confidential_information", "survival_of_obligations"]

In [ ]:
import re
import numpy as np
import torch

JUDGE_MODELS = {
    # name: HF id, residual-stream layers we cache (index into hidden_states: 0 = embeddings)
    "gemma3-4b": dict(hf_id="google/gemma-3-4b-it", layers=list(range(2, 35, 2))),     # 34 blocks -> 17 layers
    "qwen0.5b":  dict(hf_id="Qwen/Qwen2.5-0.5B-Instruct", layers=[3, 6, 9, 12, 15, 18, 21, 24]),  # 24 blocks
}
SYSTEM_PROMPT = "You are a compliance auditor."

def judge_messages(assertion, excerpt):
    """Rule = LegalBench assertion, evidence = contract excerpt. One chat turn, answer = next token."""
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Contract clause:\n{excerpt}\n\n"
                                        f"Does this clause state that {assertion}? Answer Yes or No."}]

CONF_QUESTION = "How confident are you that your answer is correct? Reply with only a number from 0 to 100."

def load_judge(choice, device, dtype=None):
    from transformers import AutoTokenizer
    hf_id = JUDGE_MODELS[choice]["hf_id"]
    tok = AutoTokenizer.from_pretrained(hf_id)
    tok.padding_side = "left"                      # last prompt token sits at position -1 for every row
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if dtype is None:   # bf16 where the GPU supports it natively (L4/A100); fp16 on a T4; bf16 on CPU (half the RAM)
        if device == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
        else:
            dtype = torch.bfloat16
    if choice.startswith("gemma"):
        from transformers import Gemma3ForConditionalGeneration as Cls   # the 4B IT checkpoint is multimodal
    else:
        from transformers import AutoModelForCausalLM as Cls
    model = Cls.from_pretrained(hf_id, dtype=dtype).to(device).eval()
    yes_id = tok.encode("Yes", add_special_tokens=False)[0]
    no_id = tok.encode("No", add_special_tokens=False)[0]
    return model, tok, yes_id, no_id

def _encode(tok, conversations, device):
    texts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversations]
    return tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)

@torch.no_grad()
def judge_batch(model, tok, yes_id, no_id, assertions, excerpts, layers=None):
    """One forward pass, no generation. Returns p_yes = P(Yes)/(P(Yes)+P(No)) at the answer position,
    the probability mass on {Yes, No}, and (optionally) the residual stream at the last prompt token."""
    enc = _encode(tok, [judge_messages(a, x) for a, x in zip(assertions, excerpts)], model.device)
    out = model(**enc, output_hidden_states=layers is not None, logits_to_keep=1)
    lp = out.logits[:, -1].float().log_softmax(-1)
    p_yes = torch.sigmoid(lp[:, yes_id] - lp[:, no_id]).cpu().numpy()
    mass = (lp[:, yes_id].exp() + lp[:, no_id].exp()).cpu().numpy()
    hidden = None
    if layers is not None:
        hidden = torch.stack([out.hidden_states[l][:, -1] for l in layers], 1).float().cpu().numpy()
    return p_yes, mass, hidden

@torch.no_grad()
def verbalized_confidence(model, tok, assertions, excerpts, verdicts, max_new_tokens=4):
    """Second turn: the judge's own verdict is in context; ask for 0-100 confidence (greedy)."""
    convs = [judge_messages(a, x) + [{"role": "assistant", "content": v},
                                     {"role": "user", "content": CONF_QUESTION}]
             for a, x, v in zip(assertions, excerpts, verdicts)]
    enc = _encode(tok, convs, model.device)
    gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    raw = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    vals = []
    for r in raw:
        m = re.search(r"\d{1,3}", r)
        vals.append(min(100.0, float(m.group())) / 100 if m else np.nan)
    return np.array(vals, dtype=np.float32), [r.strip() for r in raw]

# Grounding check helpers ------------------------------------------------------------------
IRRELEVANT_CLAUSES = [   # invented boilerplate, unrelated to any of the assertions
    "This Agreement shall be governed by and construed in accordance with the laws of the State of New York.",
    "All notices under this Agreement shall be in writing and delivered to the addresses set out above.",
    "This Agreement may be executed in counterparts, each of which shall be deemed an original.",
]

def split_segments(text):
    """Split an excerpt into sentence/clause segments (on . ; and enumerations like (a), (ii))."""
    parts = re.split(r"(?<=[.;])\s+|\s+(?=\(?[a-z0-9ivx]{1,4}\)\s)", text.strip())
    return [p.strip() for p in parts if len(p.strip()) > 3]

def truncate(text, max_chars):
    text = " ".join(str(text).split())
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + " ..."

In [ ]:
from datasets import load_dataset

def doc_split(doc):
    """Split by NDA (document_name), not by row: the same NDA appears under many assertions."""
    h = int(hashlib.md5(doc.encode()).hexdigest(), 16) % 4
    return ["train", "train", "cal", "test"][h]

rows = []
for task, assertion in ASSERTIONS.items():
    ds = load_dataset("nguha/legalbench", f"contract_nli_{task}", split="test").to_pandas()
    rng = np.random.default_rng(SEED + sum(map(ord, task)))
    yes, no = ds[ds.answer == "Yes"], ds[ds.answer == "No"]
    n_yes = min(len(yes), N_PER_ASSERTION // 2)
    n_no = min(len(no), N_PER_ASSERTION - n_yes)
    pick = pd.concat([yes.iloc[rng.permutation(len(yes))[:n_yes]], no.iloc[rng.permutation(len(no))[:n_no]]])
    for _, r in pick.iterrows():
        rows.append(dict(uid=f"{task}:{r['index']}", task=task, assertion=assertion,
                         excerpt=truncate(r["text"], MAX_EXCERPT_CHARS), document=r["document_name"],
                         gold=int(r["answer"] == "Yes"),
                         split="shift" if task in HELDOUT else doc_split(r["document_name"])))
df = pd.DataFrame(rows).reset_index(drop=True)
df.to_csv(f"{OUT_DIR}/selection.csv", index=False)
print(len(df), "examples")
print(df.groupby("split").agg(n=("gold", "size"), yes_rate=("gold", "mean"), tasks=("task", "nunique")))

## 3 · Load the judge and sanity-check it

In [ ]:
t0 = time.time()
model, tok, YES_ID, NO_ID = load_judge(MODEL_CHOICE, device)
LAYERS = JUDGE_MODELS[MODEL_CHOICE]["layers"]
print(f"loaded {JUDGE_MODELS[MODEL_CHOICE]['hf_id']} in {time.time()-t0:.0f}s; dtype={model.dtype}; caching layers {LAYERS}")
print("Yes/No token ids:", YES_ID, NO_ID, tok.convert_ids_to_tokens([YES_ID, NO_ID]))
print(tok.apply_chat_template(judge_messages("<RULE>", "<EVIDENCE>"), tokenize=False, add_generation_prompt=True))

p, mass, h = judge_batch(model, tok, YES_ID, NO_ID, df.assertion[:4].tolist(), df.excerpt[:4].tolist(), LAYERS)
print("p_yes", p.round(3), "| Yes/No mass", mass.round(3), "| hidden", h.shape, "finite:", np.isfinite(h).all())
if not (np.isfinite(p).all() and np.isfinite(h).all()) and device == "cuda":
    # Gemma models can overflow float16 activations; bfloat16 has float32's range (emulated, slower, on a T4)
    print("⚠ non-finite values in float16 → reloading in bfloat16")
    del model; torch.cuda.empty_cache()
    model, tok, YES_ID, NO_ID = load_judge(MODEL_CHOICE, device, dtype=torch.bfloat16)
    p, mass, h = judge_batch(model, tok, YES_ID, NO_ID, df.assertion[:4].tolist(), df.excerpt[:4].tolist(), LAYERS)
    print("p_yes", p.round(3), "| finite:", np.isfinite(h).all())
assert np.isfinite(p).all() and np.isfinite(h).all(), "Non-finite judge outputs: try dtype=torch.float32 (needs more memory)"

## 4 · Run the judge over all examples (one forward pass each, no generation)

In [ ]:
t0 = time.time()
P, M, H = [], [], []
order = np.argsort(df.excerpt.str.len().values)          # length-sorted batches = less padding
for i in range(0, len(df), BATCH_SIZE):
    idx = order[i:i + BATCH_SIZE]
    p, m, h = judge_batch(model, tok, YES_ID, NO_ID, df.assertion.values[idx].tolist(),
                          df.excerpt.values[idx].tolist(), LAYERS)
    P.append((idx, p, m, h))
    if (i // BATCH_SIZE) % 20 == 0:
        print(f"{i+len(idx)}/{len(df)}  {time.time()-t0:.0f}s")
p_yes = np.zeros(len(df), np.float32); mass = np.zeros(len(df), np.float32)
hidden = np.zeros((len(df), len(LAYERS), P[0][3].shape[-1]), np.float16)
n_clipped = 0
for idx, p, m, h in P:
    n_clipped += int((np.abs(h) > 65000).sum())
    p_yes[idx], mass[idx], hidden[idx] = p, m, np.clip(h, -65000, 65000).astype(np.float16)   # float16 storage
if n_clipped:
    print(f"note: {n_clipped} activation values clipped to the float16 range")
bad = ~np.isfinite(p_yes)
assert not bad.any(), (f"{bad.sum()} examples gave non-finite P(Yes) (float16 overflow?). "
                       "Re-run from section 3 with load_judge(MODEL_CHOICE, device, dtype=torch.bfloat16).")
df["p_yes"], df["yes_no_mass"] = p_yes, mass
df["verdict"] = (df.p_yes > 0.5).astype(int)
df["correct"] = (df.verdict == df.gold).astype(int)
T_JUDGE = time.time() - t0
print(f"judge pass: {T_JUDGE:.0f}s ({T_JUDGE/len(df):.2f}s/example)")
print("accuracy by split:", df.groupby("split").correct.mean().round(3).to_dict())
print("Yes rate (verdicts):", df.verdict.mean().round(3), "| mean Yes/No mass:", mass.mean().round(3))
print(df.groupby("task").correct.mean().round(2).sort_values())

**Check before continuing:** overall accuracy should be clearly above 50 % and clearly below 100 % (the demo needs both right and wrong verdicts; with Qwen 0.5B we measured 67.7 % overall, 52–92 % per assertion). A Yes-rate near 0 or 1 means the judge collapsed to one answer: change the prompt in `judge_messages` (and in the student notebook generator, which shares it).

## 5 · Verbalized confidence (second, short generation)

In [ ]:
t0 = time.time()
verbal = np.full(len(df), np.nan, np.float32); verbal_raw = np.array([""] * len(df), dtype=object)
if DO_VERBAL:
    vb = BATCH_SIZE * 2 if device == "cuda" else BATCH_SIZE
    for i in range(0, len(df), vb):
        idx = order[i:i + vb]
        v, raw = verbalized_confidence(model, tok, df.assertion.values[idx].tolist(), df.excerpt.values[idx].tolist(),
                                       ["Yes" if x else "No" for x in df.verdict.values[idx]])
        verbal[idx], verbal_raw[idx] = v, raw
        if (i // vb) % 20 == 0:
            print(f"{i+len(idx)}/{len(df)}  {time.time()-t0:.0f}s")
df["verbal_conf"], df["verbal_raw"] = verbal, verbal_raw
T_VERBAL = time.time() - t0
print(f"verbalized confidence: {T_VERBAL:.0f}s; unparsable: {np.isnan(verbal).mean():.1%}")
print(df.verbal_conf.describe().round(3))
print("examples:", df.verbal_raw.head(8).tolist())

## 6 · Grounding check (precomputed for participants without a GPU)

For the most confident **Yes** ("compliant") verdicts on unseen documents (`test` + `shift`):
- **irrelevant evidence:** replace the excerpt by an unrelated boilerplate clause → a grounded judge should now say No;
- **leave one segment out:** remove each sentence/clause in turn → which segment does the Yes depend on?

In [ ]:
t0 = time.time()
cand = df[df.split.isin(["test", "shift"]) & (df.verdict == 1)].sort_values("p_yes", ascending=False).head(N_GROUNDING)
G = dict(g_index=[], g_p_full=[], g_p_irrelevant=[], g_n_segments=[], g_p_drop_min=[], g_drop_segment=[])
for k, (i, r) in enumerate(cand.iterrows()):
    irr = IRRELEVANT_CLAUSES[k % len(IRRELEVANT_CLAUSES)]
    p_irr = judge_batch(model, tok, YES_ID, NO_ID, [r.assertion], [irr])[0][0]
    segs = split_segments(r.excerpt)
    if len(segs) > 1:
        variants = [" ".join(segs[:j] + segs[j + 1:]) for j in range(len(segs))]
        pd_ = np.concatenate([judge_batch(model, tok, YES_ID, NO_ID, [r.assertion] * len(variants[s:s + BATCH_SIZE]),
                                          variants[s:s + BATCH_SIZE])[0] for s in range(0, len(variants), BATCH_SIZE)])
        j = int(np.argmin(pd_)); p_min, seg = float(pd_[j]), segs[j]
    else:
        p_min, seg = np.nan, ""
    for key, val in zip(G, [i, r.p_yes, p_irr, len(segs), p_min, seg]):
        G[key].append(val)
T_GROUND = time.time() - t0
g = pd.DataFrame(G)
print(f"grounding: {len(g)} verdicts in {T_GROUND:.0f}s")
print("still Yes with irrelevant evidence:", (g.g_p_irrelevant > 0.5).mean().round(3))
print("flips to No when one segment is removed:", (g.g_p_drop_min < 0.5).mean().round(3),
      f"(of {np.isfinite(g.g_p_drop_min).sum()} multi-segment excerpts)")

## 7 · Save the cache (compact `.npz`, no pickle)

In [ ]:
cache_path = f"{OUT_DIR}/cache_{MODEL_CHOICE}.npz"
meta = dict(model=JUDGE_MODELS[MODEL_CHOICE]["hf_id"], model_choice=MODEL_CHOICE, layers=LAYERS,
            dtype=str(model.dtype), device=device, n=len(df), seed=SEED, n_per_assertion=N_PER_ASSERTION,
            max_excerpt_chars=MAX_EXCERPT_CHARS, heldout=HELDOUT, system_prompt=SYSTEM_PROMPT,
            prompt_template=judge_messages("{assertion}", "{excerpt}")[1]["content"], conf_question=CONF_QUESTION,
            source="nguha/legalbench contract_nli_* (test split), CC BY 4.0; ContractNLI (Koreeda & Manning 2021)",
            seconds=dict(judge=round(T_JUDGE), verbal=round(T_VERBAL), grounding=round(T_GROUND)),
            accuracy=df.groupby("split").correct.mean().round(4).to_dict(),
            created=time.strftime("%Y-%m-%d %H:%M"))
S = lambda col: np.array(df[col].astype(str).tolist())          # fixed-width unicode, loads without pickle
np.savez_compressed(
    cache_path,
    uid=S("uid"), task=S("task"), assertion=S("assertion"), excerpt=S("excerpt"), split=S("split"),
    gold=df.gold.values.astype(np.int8), verdict=df.verdict.values.astype(np.int8),
    p_yes=df.p_yes.values.astype(np.float32), yes_no_mass=df.yes_no_mass.values.astype(np.float32),
    verbal_conf=df.verbal_conf.values.astype(np.float32), verbal_raw=S("verbal_raw"),
    hidden=hidden, layers=np.array(LAYERS, np.int16),
    g_index=np.array(G["g_index"], np.int32), g_p_full=np.array(G["g_p_full"], np.float32),
    g_p_irrelevant=np.array(G["g_p_irrelevant"], np.float32), g_n_segments=np.array(G["g_n_segments"], np.int16),
    g_p_drop_min=np.array(G["g_p_drop_min"], np.float32), g_drop_segment=np.array([str(s) for s in G["g_drop_segment"]] or [""]),
    meta=np.array(json.dumps(meta)),
)
print(cache_path, f"{os.path.getsize(cache_path)/1e6:.1f} MB")
print(json.dumps(meta, indent=1))

## 8 · Upload to the Hugging Face dataset repo

In [ ]:
if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(CACHE_REPO, repo_type="dataset", exist_ok=True, private=False)
    readme = (f"# MICAI 2026 NB5 judge cache\n\nVerdicts, confidences and residual-stream activations of "
              f"`{meta['model']}` acting as a compliance judge on LegalBench contract_nli examples.\n\n"
              "Source data: LegalBench (Guha et al., 2023) `contract_nli_*`, derived from ContractNLI "
              "(Koreeda & Manning, 2021); CC BY 4.0. Excerpts truncated to "
              f"{MAX_EXCERPT_CHARS} characters. Built with notebooks/instructor/NB5_prepare_cache.ipynb.\n")
    open(f"{OUT_DIR}/README.md", "w").write("---\nlicense: cc-by-4.0\n---\n" + readme)
    for f in [cache_path, f"{OUT_DIR}/selection.csv", f"{OUT_DIR}/README.md"]:
        api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=CACHE_REPO, repo_type="dataset")
    print(f"uploaded → https://huggingface.co/datasets/{CACHE_REPO}")
else:
    print("UPLOAD is off (CACHE_REPO still a TODO). The cache is at", cache_path)
    if IN_COLAB:
        print("Download it from the Files panel, or: from google.colab import files; files.download(cache_path)")

**Time budget (fill in after your run):** model download ___ · judge pass ___ · verbalized ___ · grounding ___.
Our CPU reference run (Qwen 0.5B, 2 cores, shared machine): see `meta['seconds']` in `data/nb5/cache_qwen0.5b.npz`.